<a href="https://colab.research.google.com/github/ileeee-sh/SeSAC_29CM_Project/blob/main/29cm_Top10_Preprocessing%26SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/haven-jeon/PyKoSpacing.git --quiet
!pip install soynlp emoji openpyxl scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import emoji, re, ast, requests
from pykospacing import Spacing
from soynlp.normalizer import emoticon_normalize
from soynlp.tokenizer import LTokenizer
from soynlp.word import WordExtractor
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f'{df.shape[0]}행 로드')
print('컬럼:', list(df.columns))

# C열 = review_text
text_col = 'review_text'
df[text_col] = df[text_col].astype(str).fillna('')
df.head(3)

In [ ]:
def extract_emojis(text):
    return [c for c in text if c in emoji.EMOJI_DATA]

def remove_emojis(text):
    return ''.join(c for c in text if c not in emoji.EMOJI_DATA)

spacing = Spacing()
print('PyKoSpacing 로딩 완료')

In [ ]:
sentences = df[text_col].tolist()

word_extractor = WordExtractor(min_frequency=5)
word_extractor.train(sentences)
words = word_extractor.extract()
cohesion_scores = {w: s.cohesion_forward for w, s in words.items()}
tokenizer = LTokenizer(scores=cohesion_scores)

print(f'LTokenizer 학습 완료: {len(cohesion_scores):,}개 단어')

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

def preprocess(text):
    if not isinstance(text, str) or text.strip() == '':
        return {'emojis': [], 'spaced': text, 'normalized': text, 'tokens': []}
    emojis = extract_emojis(text)
    text_no_emoji = remove_emojis(text).strip()
    try:
        spaced = spacing(text_no_emoji) if text_no_emoji else text_no_emoji
    except:
        spaced = text_no_emoji
    normalized = emoticon_normalize(spaced, num_repeats=2)
    tokens = tokenizer.tokenize(normalized, flatten=True)
    return {'emojis': emojis, 'spaced': spaced, 'normalized': normalized, 'tokens': tokens}

results = df[text_col].progress_apply(preprocess)

df['emojis']     = results.apply(lambda x: x['emojis'])
df['spaced']     = results.apply(lambda x: x['spaced'])
df['normalized'] = results.apply(lambda x: x['normalized'])
df['tokens']     = results.apply(lambda x: x['tokens'])

print('전처리 완료')
df[['review_text', 'tokens', 'emojis']].head(3)

In [ ]:
emoji_lexicon = {
    '👍': 1.0, '❤️': 1.0, '💖': 1.0, '💕': 1.0, '💗': 1.0, '💓': 1.0,
    '🤍': 1.0, '🧡': 1.0, '💛': 1.0, '💚': 1.0, '💙': 1.0, '💜': 1.0,
    '🖤': 1.0, '💝': 1.0, '💞': 1.0, '💟': 1.0, '💯': 1.0,
    '😍': 0.95, '⭐': 0.92, '✨': 0.88, '🥰': 0.85, '😁': 0.83,
    '😊': 0.82, '😚': 0.82, '😂': 0.80, '🙌': 0.78, '😘': 0.75,
    '✅': 0.75, '🚀': 0.75, '😌': 0.75, '🎉': 0.70, '😸': 0.70, '🐵': 0.60,
    '👎': -1.0, '💔': -1.0, '😡': -0.95,
    '🤢': -0.90, '🚨': -0.90, '😭': -0.90, '❗': -0.85, '🛑': -0.85,
    '😱': -0.85, '🤬': -0.85, '😤': -0.80,
    '😣': -0.80, '😰': -0.75, '😩': -0.75, '😓': -0.70, '❌': -0.75,
    '⏳': -0.60, '🔄': -0.45, '⌛': -0.50,
    '🙄': -0.65, '😒': -0.65, '😥': -0.65,
    '📱': 0.0, '⚙️': 0.0, '🔧': -0.20, '📶': 0.15, '🔋': 0.10,
    '🔔': 0.10, '🔕': -0.20, '📴': -0.35
}
print(f'이모지 사전: {len(emoji_lexicon)}개')

In [ ]:
knu_url = "https://raw.githubusercontent.com/park1200656/KnuSentiLex/master/SentiWord_Dict.txt"
response = requests.get(knu_url)
response.encoding = 'utf-8'

rows = []
for line in response.text.strip().split('\n'):
    parts = line.strip().split('\t')
    if len(parts) == 2:
        try:
            rows.append({'word': parts[0].strip(), 'score': float(parts[1].strip()) / 2.0})
        except ValueError:
            pass

knu_lexicon = {r['word']: r['score'] for r in rows}

# 커스텀 사전 추가
custom_lexicon = {
    '좋아요': 1.0, '좋아': 0.9, '좋네요': 0.9, '좋습니다': 1.0,
    '굿': 0.9, '굳': 0.9, '굿굿': 1.0, '최고예요': 1.0, '최고': 0.9,
    '이뻐요': 1.0, '이뻐': 0.9, '이쁘다': 0.9, '이쁘네요': 0.9, '이쁜': 0.8,
    '예뻐요': 1.0, '예뻐': 0.9, '예쁘다': 0.9, '예쁜': 0.8,
    '맘에들어요': 1.0, '맘에들어': 0.9, '마음에들어요': 1.0,
    '완벽해요': 1.0, '만족해요': 0.9, '만족합니다': 0.9,
    '추천해요': 0.9, '추천합니다': 0.9, '강추': 1.0,
    '감사해요': 0.8, '감사합니다': 0.8,
    '편해요': 0.8, '편하다': 0.8, '편합니다': 0.8,
    '빨라요': 0.7, '빠르다': 0.7, '친절해요': 0.8, '친절합니다': 0.8,
    '깔끔해요': 0.8, '깔끔하다': 0.8, '행복해요': 0.9, '기뻐요': 0.9,
    '최애': 1.0, '덕분에': 0.8, '갬성': 0.7, '갠춘': 0.6, '괜츈': 0.6,
    '할인': 0.65, '쿠폰': 0.65,
    '별로에요': -0.8, '별로': -0.7, '실망이에요': -0.9, '실망했어요': -0.9, '실망': -0.8,
    '그따구로': -1.0, '글러먹은': -1.0, '무책임한': -1.0, '먹튀앱': -1.0, '성가셔': -0.7,
    '불편해요': -0.8, '불편하다': -0.8, '느려요': -0.7, '느리다': -0.7,
    '비싸요': -0.6, '비싸다': -0.6, '최악이에요': -1.0, '최악': -1.0,
    '아쉬워요': -0.6, '아쉽다': -0.6, '별로네요': -0.8, '그냥그래요': -0.4,
    '다만': -0.2, '결국': -0.2, '제발': -0.5,
    'X같이': -1.0, '렉': -0.6, '안': -0.75, '버벅': -0.6, '버벅거': -0.6,
    '잣': -1.0, '개떡': -1.0, '오류': -0.8, '에러': -0.8, '에휴': -0.9,
    '불량': -0.8, '삭제': -0.8, '벌레': -0.7, '개그지': -0.7,
    '싫어': -0.8, '꺼져': -0.7, '강제': -0.7, '쓰레기': -0.8,
}
knu_lexicon.update(custom_lexicon)
print(f'KNU + 커스텀: {len(knu_lexicon):,}개')

In [ ]:
W_KNU, W_EMOJI = 1.5, 0.35

def compute_sentiment(tokens, emojis):
    weighted_sum, weight_total = 0.0, 0.0
    matched = {'knu': [], 'emoji': []}

    for token in tokens:
        if token in knu_lexicon:
            score = knu_lexicon[token]
            weighted_sum += score * W_KNU
            weight_total += W_KNU
            matched['knu'].append((token, round(score, 3)))

    for em in emojis:
        if em in emoji_lexicon:
            score = emoji_lexicon[em]
            weighted_sum += score * W_EMOJI
            weight_total += W_EMOJI
            matched['emoji'].append((em, round(score, 3)))

    polarity = 0.0 if weight_total == 0 else max(-1.0, min(1.0, weighted_sum / weight_total))
    return polarity, matched

polarities, matched_all = [], []
for i in tqdm(range(len(df)), desc='감성분석'):
    tokens = df['tokens'].iloc[i] if isinstance(df['tokens'].iloc[i], list) else ast.literal_eval(str(df['tokens'].iloc[i]))
    emojis = df['emojis'].iloc[i] if isinstance(df['emojis'].iloc[i], list) else ast.literal_eval(str(df['emojis'].iloc[i]))
    polarity, matched = compute_sentiment(tokens, emojis)
    polarities.append(round(polarity, 4))
    matched_all.append(matched)

df['polarity']       = polarities
df['matched_words']  = matched_all
df['sentiment_label'] = df['polarity'].apply(
    lambda p: '긍정' if p >= 0.1 else ('부정' if p <= -0.1 else '중립')
)

print('감성분석 완료')
print(df['sentiment_label'].value_counts())
print(f'평균 polarity: {df["polarity"].mean():.4f}')

In [ ]:
out = df.copy()
out['matched_words'] = out['matched_words'].astype(str)
out_path = '29cm_TOP10_sentiment.csv'
out.to_csv(out_path, index=False, encoding='utf-8-sig')
files.download(out_path)
print(f'저장 완료: {out_path}')